In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')
small_llm = ChatOpenAI(model='gpt-4o-mini')

In [3]:
from langchain_core.tools import tool

@tool  
def add (a: int, b: int) -> int:  
    """숫자 a와 b를 더한 결과를 반환합니다."""   
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱한 결과를 반환합니다."""
    return a * b

In [4]:
!uv pip install -qU langchain-google-community\[gmail\]

In [5]:
from langchain_google_community import GmailToolkit


from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)

# Can review scopes here https://developers.google.com/gmail/api/auth/scopes
# For instance, readonly scope is 'https://www.googleapis.com/auth/gmail.readonly'
credentials = get_gmail_credentials(   
    token_file="./google/gmail_token.json",
    scopes=["https://mail.google.com/"],
    client_sercret_file="./google/gmail_credentials.json",
)
api_resource = build_resource_service(credentials=credentials)
gmail_toolkit = GmailToolkit(api_resource=api_resource)
gmail_toolkit_list = gmail_toolkit.get_tools()

/var/folders/dl/kpb27tds2xvfd5lcdc8n81c40000gn/T/ipykernel_24020/2605579340.py:11: DeprecationWarning: get_gmail_credentials is deprecated and will be removed in a future version.Use get_google_credentials instead.
  credentials = get_gmail_credentials(
/var/folders/dl/kpb27tds2xvfd5lcdc8n81c40000gn/T/ipykernel_24020/2605579340.py:16: DeprecationWarning: build_resource_service is deprecated and will be removed in a future version.Use build_gmail_service instead.
  api_resource = build_resource_service(credentials=credentials)


In [6]:
!uv pip install -qU duckduckgo-search langchain-community ddgs

In [7]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()


In [8]:
!uv pip install -qU langchain-community arxiv

In [10]:
from langchain_community.agent_toolkits.load_tools import load_tools

loaded_tool_list = load_tools(
    ['arxiv'],
    
    )

In [11]:
from langgraph.prebuilt import ToolNode

tool_list = [add, multiply, search_tool] + gmail_toolkit_list + loaded_tool_list
llm_with_tools = small_llm.bind_tools(tool_list) 
tool_node = ToolNode(tool_list)

In [ ]:
from langgraph.graph import MessagesState, StateGraph

class AgentState(MessagesState):
    summary: str

graph_builder = StateGraph(AgentState)

In [ ]:
from langchain_core.messages import SystemMessage

def agent(state: AgentState):
    messages = state['messages']
    summary = state['summary']
    
    if summary != '':
        messages = [SystemMessage(content=f'summarize this chat history below: \n\nchat_history:{messages}\n\nsummary:{summary}')] + messages
    response = llm_with_tools.invoke(messages)

    return {'messages': [response]}

In [ ]:
from typing import Literal
from langgraph.graph import END

def should_continue(state: AgentState) -> Literal['tools', END]:
    # 상태에서 메시지를 추출합니다.
    messages = state['messages']
    
    # 마지막 AI 메시지를 가져옵니다.
    last_ai_message = messages[-1]
    
    # 마지막 AI 메시지가 도구 호출을 포함하고 있는지 확인합니다.
    if last_ai_message.tool_calls:
        # 도구 호출이 있으면 'tools'를 반환합니다.
        return 'tools'
    
    # 도구 호출이 없으면 END를 반환하여 프로세스를 종료합니다.
    return END

In [ ]:
def summarize_messages(state: AgentState) :
    messages = state['messages']
    summary = state['summary']
    
    summary_prompt = f'summarize this chat history below: \n\nchat_history:{messages}'
    
    if summary != '':
        summary_prompt = f'''summarize this chat history below while looking at the summary of earlier conversations
                            chat_history:{messages}
                            summary:{summary}'''
                                        
    summary = small_llm.invoke(summary_prompt)
    
    # 요약된 메시지를 반환합니다.
    return {'summary': summary.content}


In [ ]:
from langchain_core.messages import RemoveMessage

def delete_messages(state: AgentState) :
    messages = state['messages']
    delete_messages = [RemoveMessage(id = message.id) for message in messages[:-3]]
    return {'messages' : delete_messages}

In [ ]:
def should_continue(state: AgentState) :
    messages = state['messages']
    last_ai_message = messages[-1]
    if last_ai_message.tool_calls:
        return 'tools'
    
    return 'delete_messages'

SyntaxError: incomplete input (4203640337.py, line 1)

In [ ]:
graph_builder.add_node('agent', agent)
graph_builder.add_node('tools', tool_node)
graph_builder.add_node('delete_messages', delete_messages)
graph_builder.add_node(summarize_messages)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_edge(START, 'agent')
graph_builder.add_conditional_edges(
    'agent',
    should_continue,
    ['tools', 'summarize_messages']
)
graph_builder.add_edge('tools', 'agent')
graph_builder.add_edge('summarize_messages', 'delete_messages')
graph_builder.add_edge('delete_messages', END)

In [18]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

graph= graph_builder.compile(
    checkpointer=checkpointer
)

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage
#query = '윤석열이가 태어난 도시의 화폐 단위는 무엇인가요? 찾아서 ujk6073@gmail.com에 메일 전송해주세요'
query = '양자컴퓨터에 관련된 논문을찾아서 이메일 초안을 작성해주세요'

config = {
    'configurable':{
        'thread_id': 'paper_summary'
    }
}
for chunk in graph.stream({'messages': [HumanMessage(query)]}, stream_mode='values'):
    chunk['messages'][-1].pretty_print()

NameError: name 'graph' is not defined

# 히스토리 삭제
1. manual 수동으로 삭제 일일이.
2. node   
 

In [3]:
current_message_list = graph.get_state(config).values['messages']
current_message_list

NameError: name 'graph' is not defined

In [ ]:
from langchain_core.messages import RemoveMessage

for message, index in enumerate(current_message_list):
    if index < len(current_message_list) - 1 :
        graph.update_state(config, {'messages': [RemoveMessage(id = message.id)]})

In [ ]:
current_message_list = graph.get_state(config).values['messages']
current_message_list